In [79]:
import pyautogui
import pandas as pd
import numpy as np
from importlib import reload
from source import gui_select


## Read in telemetry file

### **select file**

In [92]:
reload(gui_select)

ifile = gui_select.single_file()

ifile

'C:/Users/Nortek1000/Desktop/toolbox/data/recorded/test2.txt'

### **read data into dictionary**

In [93]:
import re
from collections import defaultdict

data = defaultdict(lambda: defaultdict(list))

with open(ifile, 'r') as f:
    for line in f:
        try:
            match = re.match(r'\$(\w+),(.*)\*(\w+)', line.strip())
            if not match:
                continue
            tm_type, content = match.group(1), match.group(2)
            fields = {}
            for field in content.split(','):
                if '=' in field:
                    k, v = field.split('=', 1)
                    fields[k] = v
            for key, value in fields.items():
                data[tm_type][key].append(value)
        except Exception as e:
            print(f"Skipping line due to error: {line.strip()}\n{e}")

data.keys()

dict_keys(['PNORI2', 'PNORS2', 'PNORC2'])

### **check contents**

In [88]:
pd.DataFrame(data['PNORC2'])

,DATE,TIME,CN,CP,VE,VN,VU,VU2,A1,A2,A3,A4,C1,C2,C3,C4
0,063025,104705,1,1.2,-32.768,-32.768,-32.768,-32.768,31.7,33.3,32.1,33.3,9,10,7,27
1,063025,104705,2,2.2,-32.768,-32.768,-32.768,-32.768,31.7,33.0,31.9,33.2,8,11,9,28
2,063025,104705,3,3.2,-32.768,-32.768,-32.768,-32.768,31.7,33.0,31.9,33.3,11,11,8,26
3,063025,104705,4,4.2,-32.768,-32.768,-32.768,-32.768,31.9,33.0,31.9,33.3,9,12,6,27
4,063025,104705,5,5.2,-32.768,-32.768,-32.768,-32.768,31.7,33.0,31.9,33.3,7,8,6,29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
179,063025,105045,4,4.2,-32.768,-32.768,-32.768,-32.768,31.7,32.8,32.1,33.3,11,15,7,29
180,063025,105045,5,5.2,-32.768,-32.768,-32.768,-32.768,31.9,32.6,31.9,33.3,13,17,7,30
181,063025,105045,6,6.2,-32.768,-32.768,-32.768,-32.768,31.5,32.6,31.7,33.3,11,13,11,29
182,063025,105045,7,7.2,-32.768,-32.768,-32.768,-32.768,31.5,32.8,31.7,33.2,7,14,8,27


## Save and format data

### **convert to xarray.Dataset**

In [94]:
import xarray as xr
import pandas as pd

def parse_time(df):
    df['TIME'] = pd.to_datetime(df['DATE']+' '+df['TIME'], format='%m%d%y %H%M%S')
    return df.drop(columns='DATE')

try:
    # Process PNORC2
    cdf = parse_time(pd.DataFrame(data['PNORC2'])).set_index(['TIME', 'CN'])
    ds = cdf.to_xarray()
    
    # Process PNORS2
    sdf = parse_time(pd.DataFrame(data['PNORS2'])).set_index('TIME')
    for col in sdf.columns:
        ds[col] = ('TIME', sdf[col])
    
    # Add PNORI2
    for k, v in data['PNORI2'].items():
        ds[k] = v[0]

except KeyError as e:
    print(f"Missing expected keys in data: {e}")
    ds = xr.Dataset()

ds

<xarray.Dataset> Size: 5kB
Dimensions:  (TIME: 6, CN: 7)
Coordinates:
  * TIME     (TIME) datetime64[ns] 48B 2025-06-25T17:30:35 ... 2025-06-25T17:...
  * CN       (CN) object 56B '1' '2' '3' '4' '5' '6' '7'
Data variables: (12/33)
    CP       (TIME, CN) object 336B '0.6' '1.1' '1.6' ... '2.6' '3.1' '3.6'
    VE       (TIME, CN) object 336B '2.662' '2.331' '2.537' ... '2.618' '2.567'
    VN       (TIME, CN) object 336B '-0.198' '0.439' ... '-0.878' '-0.579'
    VU       (TIME, CN) object 336B '-0.394' '-1.108' ... '-0.037' '-0.575'
    VU2      (TIME, CN) object 336B '-0.398' '-1.116' ... '-0.011' '-0.569'
    A1       (TIME, CN) object 336B '34.7' '34.1' '32.4' ... '34.1' '33.7'
    ...       ...
    SN       <U6 24B '106516'
    NB       <U1 4B '4'
    NC       <U1 4B '7'
    BD       <U4 16B '0.10'
    CS       <U4 16B '0.50'
    CY       <U3 12B 'ENU'

In [95]:
import numpy as np

def assign_dtypes(ds, dtypes, fill=None):
    for col in ds.variables:
        if col in dtypes.keys():
            ds[col] = ds[col].astype(dtypes[col])
        else:
            if fill: 
                ds[col] = ds[col].astype(fill)

dtypes = dict(
    TIME = np.datetime64,
    CN = int,
    EC = int,
    IT = int,
    NB = int,
    NC = int,
    SC = str,
    SN = str,
    CY = str
)

assign_dtypes(ds, dtypes, fill=float)

ds

<xarray.Dataset> Size: 5kB
Dimensions:  (TIME: 6, CN: 7)
Coordinates:
  * TIME     (TIME) datetime64[ns] 48B 2025-06-25T17:30:35 ... 2025-06-25T17:...
  * CN       (CN) int64 56B 1 2 3 4 5 6 7
Data variables: (12/33)
    CP       (TIME, CN) float64 336B 0.6 1.1 1.6 2.1 2.6 ... 1.6 2.1 2.6 3.1 3.6
    VE       (TIME, CN) float64 336B 2.662 2.331 2.537 ... 2.294 2.618 2.567
    VN       (TIME, CN) float64 336B -0.198 0.439 -0.76 ... -0.304 -0.878 -0.579
    VU       (TIME, CN) float64 336B -0.394 -1.108 0.205 ... -0.037 -0.575
    VU2      (TIME, CN) float64 336B -0.398 -1.116 0.177 ... -0.011 -0.569
    A1       (TIME, CN) float64 336B 34.7 34.1 32.4 32.1 ... 33.5 33.7 34.1 33.7
    ...       ...
    SN       <U6 24B '106516'
    NB       int64 8B 4
    NC       int64 8B 7
    BD       float64 8B 0.1
    CS       float64 8B 0.5
    CY       <U3 12B 'ENU'

### **export to netCDF**

In [96]:
import os

filename = os.path.basename(ifile).rpartition('.')[0]
ofile = f'data/formatted/{filename}.nc'

ds.to_netcdf(ofile)